# TSIS &mdash; Lab 2
## Retrieval-Augmented Generation (RAG)

In the first lab, we gave a language model conversation history and tools. Today we will give it **relevant external knowledge**. We will build a small RAG system for a fictional software project, inspect what it retrieves, and then connect retrieval to an agent tool.

**Prerequisites:** make sure you've solved [the first lab](lab1_bis.ipynb) beforehand.


## Learning objectives

By the end of the lab you should be able to:

- explain the **indexing** and **query** stages of a RAG system;
- split **documents** into **chunks** and retrieve candidate matches based on **vector similarity**;
- distinguish **retrieval quality** (accuracy/precision) from **answer-generation quality** (the final output);
- store/query vectors and metadata in a [**vector database**](https://en.wikipedia.org/wiki/Vector_database) ([ChromaDB](https://www.trychroma.com/products/chromadb));
- ground an answer in retrieved **evidence** and make the model abstain when evidence is missing;
- expose **retrieval as a tool** that an agent can call.

RAG is an engineering pattern, with pros and cons. It can fail silently when the right evidence is not retrieved, when chunks lose context, or when the model ignores the evidence. We will explore some of these failure mods in this lab.

## What problem does RAG solve?

A model's weights are not a live database. They do not contain your private repository and they cannot learn a change made five minutes ago. RAG injects selected source material into the model's context at query time:

1. **Indexing (ahead of time):** load documents, split them into chunks, turn chunks into vectors, and store vectors plus metadata.
2. **Retrieval (per question):** turn the question into a vector and find nearby chunks.
3. **Generation:** put the question and retrieved chunks in a prompt and ask the model to answer from that evidence.

RAG trades infrastructure and a possible retrieval failure for a smaller, more focused context. Long-context prompting is a useful alternative when the complete source set is small or cross-document comparison matters. Production systems often combine both approaches, add metadata filters and reranking, and measure retrieval separately from generation.

## Setup and the local corpus

The notebook uses the same OpenAI-compatible interface as `lab1_bis.ipynb`. If you run into issues, ask the teaching assistant for help.

In [29]:
%pip install --quiet --upgrade openai scikit-learn chromadb

/mnt/projects/acc-a-fmi-26-001/materiale-tsis/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import json
from pathlib import Path

from pydantic import BaseModel

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "rag"

class Document(BaseModel):
    source: str
    text: str

documents: list[Document] = []
for path in sorted(DATA_DIR.glob("*.md")):
    document = Document(source=path.name, text=path.read_text(encoding="utf-8"))
    documents.append(document)

print(f"Loaded {len(documents)} documents:", [d.source for d in documents])
print(documents[0].text[:300])

Loaded 3 documents: ['architecture.md', 'operations.md', 'security.md']
# Atlas project architecture

Atlas is a small service that imports software-project events and exposes a
read-only reporting API. The service is split into an API process and a
background worker. The API process validates requests and reads from the
database; the worker processes imports asynchrono


## Chunking: deciding what can be retrieved

A whole document is often too broad to retrieve precisely, while tiny chunks lose context. There is no universally correct chunk size: headings, paragraphs, token counts, overlap, document structure, and the downstream question all matter.

For this small corpus, a paragraph is a useful and inspectable chunk. The metadata lets us show students exactly where an answer came from.

In [31]:
class DocumentChunk(BaseModel):
    id: str
    source: str
    chunk_index: int
    text: str


def chunk_documents(documents: list[Document]) -> list[DocumentChunk]:
    chunks = []
    for document in documents:
        paragraphs = [p.strip() for p in document.text.split("\n\n") if p.strip()]

        for index, paragraph in enumerate(paragraphs):
            chunk = DocumentChunk(
                id=f"{document.source}#{index}",
                source=document.source,
                chunk_index=index,
                text=paragraph,
            )
            chunks.append(chunk)

    return chunks

chunks: list[DocumentChunk] = chunk_documents(documents)
for chunk in chunks:
    print(chunk.id, "->", chunk.text[:100].replace("\n", " "))

architecture.md#0 -> # Atlas project architecture
architecture.md#1 -> Atlas is a small service that imports software-project events and exposes a read-only reporting API.
architecture.md#2 -> The production deployment uses PostgreSQL 16. Local development may use SQLite, but SQLite is not su
architecture.md#3 -> The repository is organized as `src/atlas/api`, `src/atlas/worker`, and `tests/`. The API and worker
operations.md#0 -> # Atlas project operations
operations.md#1 -> Deployments are performed by the release engineer after the CI pipeline is green. A deployment must 
operations.md#2 -> The service emits structured JSON logs. Every request log includes a `request_id`; import jobs also 
operations.md#3 -> The on-call engineer acknowledges a production alert within 15 minutes. If the error rate remains ab
security.md#0 -> # Atlas project security
security.md#1 -> Atlas authenticates users with the university identity provider using OpenID Connect. API access is 
security.md#2 -> S

## A transparent local vector index

To keep the first implementation understandable, we use [TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) (_term frequency&ndash;inverse document frequency_) vectors. TF-IDF is not a modern semantic embedding model: it mainly matches words and phrases. That is a feature for teaching because we can inspect the representation and avoid requiring a second model endpoint. Cosine similarity gives us a ranked list of candidate chunks.

In a production system, an embedding model is normally chosen and versioned separately from the generation model. The rest of the retrieval interface can stay largely the same.

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
matrix = vectorizer.fit_transform([chunk.text for chunk in chunks])

class SearchResult(DocumentChunk):
    score: float

def retrieve(query, k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, matrix)[0]
    ranked = scores.argsort()[::-1][:k]
    return [
        SearchResult(**chunks[index].model_dump(), score=float(scores[index]))
        for index in ranked
    ]

def show_results(results):
    for result in results:
        print(f"{result.score:.3f}  {result.id}\n{result.text}\n")

show_results(retrieve("What database is used in production?"))

0.135  architecture.md#2
The production deployment uses PostgreSQL 16. Local development may use
SQLite, but SQLite is not supported for the production deployment. Import
jobs are placed on a Redis-backed queue. The API is stateless, so multiple API
instances can run behind a load balancer.

0.101  operations.md#1
Deployments are performed by the release engineer after the CI pipeline is
green. A deployment must have a rollback plan and a link to the release
notes. Database migrations run before the new application version receives
traffic.

0.083  architecture.md#1
Atlas is a small service that imports software-project events and exposes a
read-only reporting API. The service is split into an API process and a
background worker. The API process validates requests and reads from the
database; the worker processes imports asynchronously.



**Observe** how retrieval returns candidates (raw document chunks), not formatted answers. Try a synonym or an unanswerable question and inspect the scores. A top result with a low score is not evidence that the corpus contains the answer.

**Exercise 1 (5 minutes):** change the chunking function so that adjacent short paragraphs are combined. Does the top result improve for a question that needs two facts? What context is lost when chunks become too large?

## The same index in a vector database

A vector database stores vectors together with IDs, documents, and metadata, then handles persistence, filtering, and nearest-neighbor queries. Here Chroma receives the vectors we already computed, so it does not download or choose an embedding model behind our backs. Real deployments commonly use a hosted embedding service or a local embedding model and configure an approximate-nearest-neighbor index.


In [33]:
import chromadb

chroma = chromadb.Client()
collection = chroma.get_or_create_collection(
    name="atlas-handbook",
    configuration={"hnsw": {"space": "cosine"}},
)
collection.add(
    ids=[chunk.id for chunk in chunks],
    documents=[chunk.text for chunk in chunks],
    metadatas=[{"source": chunk.source, "chunk_index": chunk.chunk_index} for chunk in chunks],
    embeddings=matrix.toarray().tolist(),
)

def retrieve_from_database(query, k=3):
    query_vector = vectorizer.transform([query]).toarray().tolist()
    result = collection.query(query_embeddings=query_vector, n_results=k)
    return [
        SearchResult(
            id=result["ids"][0][i],
            text=result["documents"][0][i],
            source=result["metadatas"][0][i]["source"],
            score=1 - result["distances"][0][i],
            chunk_index=result["metadatas"][0][i]["chunk_index"]
        )
        for i in range(len(result["ids"][0]))
    ]

show_results(retrieve_from_database("How are users authenticated?"))

0.150  security.md#1
Atlas authenticates users with the university identity provider using
OpenID Connect. API access is authorized with project-level roles. The
service must never log access tokens, passwords, or complete request bodies.

0.000  architecture.md#3
The repository is organized as `src/atlas/api`, `src/atlas/worker`, and
`tests/`. The API and worker share domain types from `src/atlas/domain`.

0.000  security.md#3
Security-sensitive changes require review from a member of the security
rotation. Dependencies are scanned weekly, and a critical dependency
vulnerability blocks a release until it is patched or explicitly accepted by
the security owner.



A production vector store is more than a Python dictionary: it usually offers durable storage, collection/index management, metadata filters, concurrent updates, and operational metrics. Examples include Chroma, pgvector, Qdrant, Milvus, and managed search services. The important design boundary is still the same: **source documents → chunks → embeddings → searchable store**.

Notice that vector similarity is not the same as truth. A system may retrieve a plausible but irrelevant chunk. A score threshold, a reranker, a keyword/metadata filter, or a request for more context can help—but each adds a design and evaluation cost.

## Grounded generation through vLLM

The prompt should tell the model what counts as evidence and what to do when the evidence is insufficient. We also include source labels so that the answer can be inspected.

In [34]:
from openai import OpenAI

BASE_URL = "http://localhost:27182/v1"
MODEL = "Qwen/Qwen3.8-27B-FP8"
client = OpenAI(api_key="EMPTY", base_url=BASE_URL)

def build_context(results):
    return "\n\n".join(
        f"[Source: {result.source}, chunk {result.chunk_index}]\n{result.text}"
        for result in results
    )

def answer_with_rag(question, k=3):
    results = retrieve_from_database(question, k=k)
    prompt = f"""You answer questions about the Atlas project.
Use only the evidence between <evidence> tags. If the evidence does not answer the question, say exactly that the handbook does not provide enough information. Do not invent details. Cite the source filename in square brackets.

<evidence>
{build_context(results)}
</evidence>

Question: {question}"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content, results

question = "How quickly must an on-call engineer acknowledge an alert?"

answer, evidence = answer_with_rag(question)
print("Question:", question)
print(answer)
print("\nRetrieved evidence:")
show_results(evidence)

Question: How quickly must an on-call engineer acknowledge an alert?


Within 15 minutes [operations.md].

Retrieved evidence:
0.289  operations.md#3
The on-call engineer acknowledges a production alert within 15 minutes. If
the error rate remains above 5 percent for 10 minutes, the on-call engineer
rolls back the application unless the incident commander chooses another
documented mitigation.

0.088  operations.md#1
Deployments are performed by the release engineer after the CI pipeline is
green. A deployment must have a rollback plan and a link to the release
notes. Database migrations run before the new application version receives
traffic.

0.000  architecture.md#3
The repository is organized as `src/atlas/api`, `src/atlas/worker`, and
`tests/`. The API and worker share domain types from `src/atlas/domain`.



**Exercise 2 (10 minutes):** ask the system about the cloud provider. Compare the answer with the retrieved chunks. Then remove the abstention instruction and ask again. What changed? This is a small example of why prompt grounding and an explicit unknown policy matter, even though neither can compensate for missing retrieval evidence.

## Evaluation and failure analysis

We can evaluate retrieval without judging prose: did the expected source appear in the top-k results? This is a tiny labelled fixture, not a statistically meaningful benchmark, but it gives us a repeatable regression check. In a real system, expand the question set, define relevance carefully, and track recall@k, precision@k, latency, cost, and answer faithfulness.

In [35]:
class Question(BaseModel):
    question: str
    expected_source: str | None
    answerable: bool

questions = json.loads((DATA_DIR / "questions.json").read_text(encoding="utf-8"))
questions = [Question(**item) for item in questions]

def retrieval_report(k=3):
    rows = []
    for item in questions:
        results = retrieve_from_database(item.question, k=k)
        sources = {result.source for result in results}
        hit = item.expected_source in sources if item.expected_source else True
        rows.append({"question": item.question, "expected": item.expected_source, "hit": hit})
    return rows

report = retrieval_report(k=3)
for row in report:
    print(f"{'PASS' if row['hit'] else 'MISS'} | expected={row['expected']} | {row['question']}")
print(f"Recall-like score: {sum(row['hit'] for row in report)}/{len(report)}")

# Inspect the unanswerable question instead of treating a top-1 result as proof.
show_results(retrieve_from_database("Which cloud provider hosts the production cluster?", k=3))

PASS | expected=architecture.md | Which database is supported in production?
PASS | expected=operations.md | How quickly must an on-call engineer acknowledge a production alert?
PASS | expected=security.md | How does Atlas authenticate users?
PASS | expected=None | Which cloud provider hosts the production cluster?
Recall-like score: 4/4
0.123  architecture.md#2
The production deployment uses PostgreSQL 16. Local development may use
SQLite, but SQLite is not supported for the production deployment. Import
jobs are placed on a Redis-backed queue. The API is stateless, so multiple API
instances can run behind a load balancer.

0.120  security.md#1
Atlas authenticates users with the university identity provider using
OpenID Connect. API access is authorized with project-level roles. The
service must never log access tokens, passwords, or complete request bodies.

0.070  operations.md#3
The on-call engineer acknowledges a production alert within 15 minutes. If
the error rate remains above 

Typical failure modes to investigate:

- **chunking failure:** the answer is split across chunks or its heading is separated from its body;
- **representation failure:** TF-IDF misses a synonym that a semantic embedding might match;
- **ranking failure:** a related but wrong chunk appears above the useful chunk;
- **coverage failure:** the source simply does not contain the answer; and
- **generation failure:** the model ignores, overgeneralizes, or contradicts supplied evidence.

**Exercise 3 (10 minutes):** change `k` between 1 and 5, and compare the retrieval report. Which question is most sensitive to `k`? In a production design, would you change chunking, embeddings, reranking, metadata filters, or the prompt first? Explain your choice.

## RAG as an agent tool

Instead of putting retrieval in every prompt-building function, an agent can decide when it needs the handbook. This is the same control boundary as Lab 1 tool calling: the model proposes a tool call, but application code executes retrieval and returns the result.

In [36]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_atlas_handbook",
        "description": "Search the Atlas project handbook for evidence relevant to a question.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]

def ask_agent(question):
    messages = [{"role": "user", "content": question}]
    first = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
    message = first.choices[0].message
    if not message.tool_calls:
        return message.content

    messages.append(message)
    for call in message.tool_calls:
        if call.function.name != "search_atlas_handbook":
            raise ValueError(f"Unexpected tool: {call.function.name}")
        arguments = json.loads(call.function.arguments)
        results = retrieve_from_database(arguments["query"], k=3)
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": build_context(results),
        })
    final = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return final.choices[0].message.content

question = "What is the rollback policy when the error rate is high?"
print("Question:", question)

# The model may answer directly or call the retrieval tool depending on its tool-choice decision.
print(ask_agent(question))

Question: What is the rollback policy when the error rate is high?


Based on the operations documentation, the rollback policy when the error rate is high is as follows:

- **Threshold:** If the error rate remains **above 5% for 10 minutes**, the on-call engineer **rolls back the application**.
- **Exception:** The rollback can be avoided only if the **incident commander chooses another documented mitigation**.
- **Acknowledgment:** The on-call engineer must acknowledge the production alert within **15 minutes**.

Additionally, every deployment is required to have a **rollback plan** and a link to the release notes before it goes live.


## Takeaways

- RAG is a pipeline: ingest, chunk, represent, retrieve, augment, and generate.
- Retrieval is an independent component that needs its own tests and metrics.
- Vector databases store vectors plus metadata; they do not make an embedding model or a prompt automatically correct.
- Grounded prompts should define evidence boundaries and an unknown/abstention behavior.
- Long-context prompting can be simpler for small, complete corpora, while RAG reduces repeated context processing and focuses attention on relevant material.
- In an agentic system, retrieval can be a controlled tool: the application owns the data access and the model decides when to request it.

For further work, replace TF-IDF with a versioned embedding model, add metadata filters and reranking, persist Chroma to a directory, and build a larger labelled evaluation set.